# The Anatomy of AI · Lecture 19 · Predicting the Next Word — the notebook

In the lecture we followed one sentence through GPT-2 small:

> *It was a warm, sunny day, so we went for a* …

GPT-2 answered **walk**, with a chance of 58%. Here you build that machine yourself, from its raw files, with no
library model:

1. **text into tokens**: the byte-pair tokenizer, from GPT-2's own two files;
2. **words as arrows**: GPT-2's word table, and word arrows you train yourself on Wikipedia;
3. **the running total**: LayerNorm, the smooth gate, the twelve blocks;
4. **from the total to chances**: one dot product per word, then softmax; this is where Plate I's 58% arrives;
5. **reading the answer as it forms**: the logit lens, and who pushes walk.

The notebook builds the pieces in the order the model uses them, so it visits the plates in a slightly different
order from the lecture; each step names its plate. Every key number is checked against the lecture's own: a ✓ means you
got what the lecture got.

It takes about **8 minutes** on free Colab with a GPU (*Runtime → Change runtime type → T4 GPU*). The GPU is needed
for Part 2's word arrows; the rest also runs on a CPU, more slowly.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3", "pyarrow==18.1.0", "regex==2024.11.6")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

In [ ]:
SENTENCE, TARGET = "It was a warm, sunny day, so we went for a", " walk"
print(SENTENCE, "…")

## Part 1 · Text into tokens

### 1. Merges, learned (Plate VI, top)
GPT-2 does not read letters or whole words. It reads **tokens**: pieces of text from a fixed list of 50,257. The list
was made by **byte-pair merging** (Sennrich et al., 2016): start from single characters, count every pair of
neighbours in a big pile of text, glue the most frequent pair into one new piece, and repeat. Common words end up as
one piece; rare words stay in several.

Here it is on the plate's toy pile: four words, each seen a few times. `_` marks the end of a word.

In [ ]:
TOY = {"low": 5, "lower": 2, "newest": 6, "widest": 3}         # word -> how often it appears

def learn_merges(corpus, n):
    words = [(list(w) + ["_"], c) for w, c in corpus.items()]    # each word as single characters, then "_"
    merges = []
    for _ in range(n):
        counts = {}                                              # every neighbouring pair, counted
        for p, c in words:
            for a, b in zip(p, p[1:]):
                counts[(a, b)] = counts.get((a, b), 0) + c
        (a, b), seen = max(counts.items(), key=lambda kv: kv[1])   # the most frequent (the first one, if tied)
        merges.append(a + b)
        new = []
        for p, c in words:                                       # glue every a, b into ab
            out, i = [], 0
            while i < len(p):
                if i < len(p) - 1 and p[i] == a and p[i + 1] == b:
                    out.append(a + b); i += 2
                else:
                    out.append(p[i]); i += 1
            new.append((out, c))
        words = new
        print(f"merge {len(merges):2d}: {a} + {b}  (seen {seen} times)    " + "   ".join(" ".join(p) for p, _ in words))
    return merges

toy = learn_merges(TOY, 10)
check("the ten merges the plate shows", toy, ['es', 'est', 'est_', 'lo', 'low', 'ne', 'new', 'newest_', 'low_', 'wi'])

### 2. GPT-2's own list (Plates V–VI)
GPT-2's list is 256 single **bytes** (every character of every language is one to four bytes) plus 50,000 merges,
learned the same way from 40 GB of web pages, plus one marker, `<|endoftext|>`. Two files hold it: `merges.txt`, the
merges in the order they were learned, and `vocab.json`, each piece's number (its **id**).

To cut text with it:
1. turn the text into bytes, and each byte into one printable character (so a space becomes `Ġ`);
2. split the text into rough words with one fixed pattern (letters, numbers, punctuation, spaces; a space stays
   attached to the word after it);
3. inside each rough word, keep gluing the neighbouring pair whose merge was learned **earliest**, until no pair is in
   the list.

In [ ]:
import regex as re                                     # like Python's re, plus \p{L} = "any letter"
HF = "https://huggingface.co/openai-community/gpt2/resolve/main/"
fetch(HF + "vocab.json", "vocab.json", "GPT-2's token list")
fetch(HF + "merges.txt", "merges.txt", "GPT-2's 50,000 merges")

VOCAB = json.load(open("vocab.json", encoding="utf-8"))           # piece -> id
ID2TOK = {i: t for t, i in VOCAB.items()}
RANK = {tuple(m.split()): r for r, m in enumerate(l for l in open("merges.txt", encoding="utf-8").read().split("\n")[1:] if l)}
print(f"{len(VOCAB):,} pieces; {len(RANK):,} merges; the first five merges: {list(RANK)[:5]}")
check("pieces in GPT-2's list", len(VOCAB), 50257)
check("merges", len(RANK), 50000)

# 1. bytes -> printable characters: the 188 printable bytes stand for themselves, the other 68 are moved past 255
_bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
_cs = _bs + [256 + n for n in range(256 - len(_bs))]
_bs = _bs + [b for b in range(256) if b not in _bs]
B2U = dict(zip(_bs, map(chr, _cs))); U2B = {u: b for b, u in B2U.items()}
# 2. the fixed pattern that cuts text into rough words
SPLIT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
_seen = {}

def bpe(piece):
    """3. glue neighbours, earliest-learned merge first, until no neighbouring pair is a merge."""
    if piece not in _seen:
        w = list(piece)
        while len(w) > 1:
            r, i = min((RANK.get(p, 1 << 30), i) for i, p in enumerate(zip(w, w[1:])))
            if r == 1 << 30:
                break
            a, b, out, j = w[i], w[i + 1], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    out.append(a + b); j += 2
                else:
                    out.append(w[j]); j += 1
            w = out
        _seen[piece] = w
    return _seen[piece]

def encode(text):
    return [VOCAB[t] for m in SPLIT.findall(text) for t in bpe("".join(B2U[b] for b in m.encode("utf-8")))]

def decode(ids):
    return bytes(U2B[ch] for i in ids for ch in ID2TOK[int(i)]).decode("utf-8", errors="replace")

In [ ]:
ids = encode(SENTENCE)
print(len(ids), "tokens:", " | ".join(decode([i]) for i in ids))
print("ids:", ids)
check("the sentence's token ids", ids, [1026, 373, 257, 5814, 11, 27737, 1110, 11, 523, 356, 1816, 329, 257])
check("the target, ' walk', is one token", encode(TARGET), [2513])

for word, pieces in [[' walk', [' walk']], [' walking', [' walking']], [' walked', [' walked']], ['walk', ['walk']], [' Walk', [' Walk']], [' interpretability', [' interpret', 'ability']], [' Dandekar', [' D', 'and', 'ek', 'ar']], [' Vizuara', [' Viz', 'u', 'ara']], [' neuron', [' neuron']], [' neurons', [' neurons']], [' unbelievable', [' unbelievable']], [' the', [' the']], [' sunny', [' sunny']], [' 2026', [' 20', '26']], [' Paris', [' Paris']]]:
    print(f"{word!r:22} -> {[decode([i]) for i in encode(word)]}")
check("all 15 of the lecture's examples cut the same way", [[decode([i]) for i in encode(w)] for w, _ in [[' walk', [' walk']], [' walking', [' walking']], [' walked', [' walked']], ['walk', ['walk']], [' Walk', [' Walk']], [' interpretability', [' interpret', 'ability']], [' Dandekar', [' D', 'and', 'ek', 'ar']], [' Vizuara', [' Viz', 'u', 'ara']], [' neuron', [' neuron']], [' neurons', [' neurons']], [' unbelievable', [' unbelievable']], [' the', [' the']], [' sunny', [' sunny']], [' 2026', [' 20', '26']], [' Paris', [' Paris']]]],
      [p for _, p in [[' walk', [' walk']], [' walking', [' walking']], [' walked', [' walked']], ['walk', ['walk']], [' Walk', [' Walk']], [' interpretability', [' interpret', 'ability']], [' Dandekar', [' D', 'and', 'ek', 'ar']], [' Vizuara', [' Viz', 'u', 'ara']], [' neuron', [' neuron']], [' neurons', [' neurons']], [' unbelievable', [' unbelievable']], [' the', [' the']], [' sunny', [' sunny']], [' 2026', [' 20', '26']], [' Paris', [' Paris']]]])

## Part 2 · Words as arrows

### 3. The weights file
All of GPT-2 small is one file of numbers, 548 MB. Its format (*safetensors*) is simple enough to read by hand: eight
bytes that say how long the table of contents is, the table of contents (JSON: each array's name, shape and place in
the file), then the numbers themselves, one after another, as 4-byte floats.

In [ ]:
fetch(HF + "model.safetensors", "gpt2.safetensors", "GPT-2 small's weights")

def read_safetensors(path):
    raw = open(path, "rb").read()
    n = int.from_bytes(raw[:8], "little")                      # how long the table of contents is
    head = json.loads(raw[8:8 + n])
    out = {}
    for name, h in head.items():
        if name != "__metadata__" and not name.endswith((".attn.bias", ".attn.masked_bias")):   # masks, not weights
            a, b = h["data_offsets"]
            out[name] = torch.from_numpy(np.frombuffer(raw, dtype=np.float32, count=(b - a) // 4, offset=8 + n + a)
                                         .reshape(h["shape"]).copy()).to(DEV)
    return out

SD = read_safetensors("gpt2.safetensors")
for name in ["wte.weight", "wpe.weight", "h.0.ln_1.weight", "h.0.attn.c_attn.weight", "h.0.attn.c_proj.weight",
             "h.0.ln_2.weight", "h.0.mlp.c_fc.weight", "h.0.mlp.c_proj.weight", "...", "ln_f.weight"]:
    print(f"  {name:24}", tuple(SD[name].shape) if name in SD else "")
n_weights = sum(v.numel() for v in SD.values())
print(f"{len(SD)} arrays, {n_weights:,} weights")
check("GPT-2 small's weights", n_weights, 124439808)

### 4. A token becomes 768 numbers (Plate VII)
The **word table** (`wte`) has one row of 768 numbers for each of the 50,257 tokens. A token's row is the first thing
the model knows about it. We call a row an **arrow**: a direction and a length in a space with 768 axes.

In [ ]:
WTE, WPE = SD["wte.weight"], SD["wpe.weight"]
print("word table:", tuple(WTE.shape))
walk = WTE[2513]
print("' walk', the first 12 of its 768 numbers:", np.round(walk[:12].tolist(), 4))
print(f"its length: {walk.norm():.3f}")
check("walk's first 12 numbers (largest difference)", (walk[:12].cpu() - torch.tensor([0.0079, -0.0973, 0.136, 0.0534, -0.139, -0.025, -0.333, -0.2829, -0.0569, -0.0572, -0.144, -0.0777])).abs().max().item(), 0, tol=1e-4)
check("walk's arrow length", walk.norm().item(), 3.219, tol=1e-3)
check("the lengths of the sentence's 13 arrows (largest difference)",
      (WTE[ids].norm(dim=-1).cpu() - torch.tensor([2.763, 2.672, 2.573, 3.284, 3.078, 3.613, 3.087, 3.078, 2.748, 2.783, 2.946, 2.486, 2.573])).abs().max().item(), 0, tol=1e-3)

fig, ax = plt.subplots(figsize=(9, 1.6))
ax.imshow(WTE[ids].cpu().numpy()[:, :96], cmap="gray_r", aspect="auto", vmin=-0.3, vmax=0.3)
ax.set_yticks(range(len(ids)), [decode([i]) for i in ids], fontsize=8); ax.set_xlabel("the first 96 of 768 numbers")
ax.set_title("The sentence's 13 rows of the word table", fontsize=10); plt.show()

### 5. Words as arrows (Plate VIII)
Two arrows point the same way when their **cosine** is near 1: the dot product of the two arrows divided by both
lengths (`a · b = a₁b₁ + a₂b₂ + …`). Which tokens point most like *walk* in GPT-2's own table?

In [ ]:
WN = WTE / WTE.norm(dim=-1, keepdim=True)                  # every arrow scaled to length 1

def nearest(i, k=8):
    sims = WN @ WN[i]
    sims[i] = -2                                           # leave the word itself out
    v, j = sims.topk(k)
    return [(decode([q]), round(x, 3)) for x, q in zip(v.tolist(), j.tolist())]

for w in (" walk", " sunny", " king"):
    print(f"{w!r:9} is nearest to", nearest(encode(w)[0]))
check("walk's four nearest arrows", nearest(2513, 4), [tuple(x) for x in [[' walking', 0.72], [' walks', 0.704], [' walked', 0.694], [' Walk', 0.694]]])
check("sunny's four nearest arrows", nearest(encode(" sunny")[0], 4), [tuple(x) for x in [[' cloudy', 0.66], [' sunshine', 0.601], [' chilly', 0.588], [' rainy', 0.587]]])

### 6. Learning arrows ourselves (Plate IX)
Where do such arrows come from? Train some yourself, with **word2vec** (Mikolov et al., 2013), on **text8**: the first
100 MB of English Wikipedia (2006), cleaned to lower-case words, 17 million of them.

The idea: a word is known by the company it keeps. Every word gets an arrow of 200 numbers. A window slides along the
text; the word in the middle is pulled toward the words up to 5 places either side of it, and pushed away from 10
words drawn at random (*negative sampling*). "Pulled toward" means: make the dot product of their arrows large, so
that σ(dot) = 1 / (1 + e^−dot) is near 1; "pushed away": near 0. (Each word really has two arrows, one for when it is
in the middle and one for when it is a neighbour; the middle ones are the word arrows.)

The lecture's settings: words seen at least 5 times, frequent words like *the* skipped at random (subsampling,
10⁻⁴), 5 passes through the text. The lecture trained with the gensim library on 16 processors; here it is written
out in PyTorch and trained in batches of 16,384 pairs on the GPU (with the Adam optimizer). Same objective, a different
route to it, so your arrows will be like the lecture's, not identical to them.

In [ ]:
import collections, zipfile
fetch("http://mattmahoney.net/dc/text8.zip", "text8.zip", "text8")
words = zipfile.ZipFile("text8.zip").read("text8").decode().split()
count = collections.Counter(words)
vocab = [w for w, n in sorted(count.items(), key=lambda kv: (-kv[1], kv[0])) if n >= 5]
index = {w: i for i, w in enumerate(vocab)}
freq = np.array([count[w] for w in vocab], dtype=np.float64)
text = torch.tensor([index[w] for w in words if w in index], device=DEV)
print(f"{len(words):,} words of text; {len(vocab):,} different words seen at least 5 times")
check("words in text8", len(words), 17005207)
check("words seen at least 5 times", len(vocab), 71290)
print("a window of real text around 'king':", " ".join(words[20258 - 8:20258 + 9]))

In [ ]:
torch.manual_seed(19)
V, DIM, WINDOW, NEG, BATCH, PASSES, LR = len(vocab), 200, 5, 10, 16384, 5, 2e-3
keep_p = torch.tensor(np.minimum(1, (np.sqrt(freq / (1e-4 * freq.sum())) + 1) * 1e-4 * freq.sum() / freq), device=DEV).float()
# the random words are drawn by frequency^0.75 (rare words a little more often than their share), from a table of
# 10 million slots in which each word fills its share, as the original word2vec does
slots = np.round(freq ** 0.75 / (freq ** 0.75).sum() * 1e7).astype(np.int64)
neg_table = torch.repeat_interleave(torch.arange(V, device=DEV), torch.tensor(slots, device=DEV))
w_in = torch.nn.Embedding(V, DIM).to(DEV); w_out = torch.nn.Embedding(V, DIM).to(DEV)
with torch.no_grad():
    w_in.weight.uniform_(-0.5 / DIM, 0.5 / DIM); w_out.weight.zero_()
opt = torch.optim.Adam([w_in.weight, w_out.weight], lr=LR)

def near(w, k=8):
    E_ = torch.nn.functional.normalize(w_in.weight.detach(), dim=1)
    v, i = (E_ @ E_[index[w]]).topk(k + 1)
    return [(vocab[j], round(x, 3)) for x, j in zip(v.tolist()[1:], i.tolist()[1:])]

def pairs():
    """One pass: skip frequent words at random, then pair every word with each neighbour inside its window
    (the window's width is drawn from 1 to 5 for every word, as word2vec does), and shuffle the pairs."""
    kept = text[torch.rand(len(text), device=DEV) < keep_p[text]]
    b = torch.randint(1, WINDOW + 1, (len(kept),), device=DEV)
    mid, nb = [], []
    for d in range(1, WINDOW + 1):
        m = b[:-d] >= d; mid.append(kept[:-d][m]); nb.append(kept[d:][m])       # the neighbour d places after
        m = b[d:] >= d; mid.append(kept[d:][m]); nb.append(kept[:-d][m])       # ... and d places before
    mid, nb = torch.cat(mid), torch.cat(nb)
    o = torch.randperm(len(mid), device=DEV)
    return mid[o], nb[o]

print("king, at the start (random arrows):", near("king", 5))
t = time.time(); step, n_steps = 0, None
for ep in range(PASSES):
    mid, nb = pairs()
    n_steps = n_steps or PASSES * math.ceil(len(mid) / BATCH)
    for s in range(0, len(mid), BATCH):
        for gr in opt.param_groups:
            gr["lr"] = LR * max(1e-4, 1 - step / n_steps)                         # ease the step size down to 0
        m, x = mid[s:s + BATCH], nb[s:s + BATCH]
        ng = neg_table[torch.randint(0, len(neg_table), (len(m) * NEG,), device=DEV)].view(len(m), NEG)
        vm = w_in(m)
        pull = torch.nn.functional.logsigmoid((vm * w_out(x)).sum(-1))                        # neighbours: σ(dot) -> 1
        push_ = torch.nn.functional.logsigmoid(-torch.bmm(w_out(ng), vm[:, :, None])[..., 0]).sum(-1)   # randoms -> 0
        loss = -(pull + push_).mean()
        opt.zero_grad(); loss.backward(); opt.step(); step += 1
    print(f"pass {ep + 1} ({len(mid) / 1e6:.0f} M pairs, {time.time() - t:.0f} s): king is nearest to", near("king", 5))

### 7. King − man + woman, and the honest twist (Plates X–XI)
The famous test: take *king*'s arrow, subtract *man*'s, add *woman*'s, and ask which word's arrow points most like the
result. The answer usually given is *queen*. The honest twist: the nearest word is almost always ***king*** itself.
*Queen* wins only when the three input words are left out of the answer.

In [ ]:
def analogy(a, b, c_, table=None, k=5):
    """Nearest words to a - b + c, by cosine: including the three inputs, and leaving them out."""
    E_ = w_in.weight.detach() if table is None else table
    ix = (lambda w: index[w]) if table is None else (lambda w: encode(w)[0])
    name = (lambda i: vocab[i]) if table is None else (lambda i: decode([i]))
    v = E_[ix(a)] - E_[ix(b)] + E_[ix(c_)]
    sims = torch.nn.functional.cosine_similarity(E_, v[None], dim=-1)
    order = sims.argsort(descending=True)[:k + 3].tolist()
    inc = [(name(i), round(sims[i].item(), 4)) for i in order[:k]]
    exc = [(name(i), round(sims[i].item(), 4)) for i in order if i not in (ix(a), ix(b), ix(c_))][:k]
    return inc, exc

inc, exc = analogy("king", "man", "woman")
print("your word2vec   including the inputs:", inc)
print("                leaving them out:    ", exc)
E_ = w_in.weight.detach()
sims = torch.nn.functional.cosine_similarity(E_, (E_[index["king"]] - E_[index["man"]] + E_[index["woman"]])[None], dim=-1)
q_rank = int((sims > sims[index["queen"]]).sum()) + 1 - sum(int(sims[index[w]] > sims[index["queen"]]) for w in ("king", "man", "woman"))
print(f"queen's place, inputs left out: {q_rank}")
check("your word2vec: the nearest word, inputs included", inc[0][0], ['king', 'queen'][0])
# our trainer is not gensim's, so its arrows differ in detail: queen within the top 3 counts as the lecture's result
check("your word2vec: queen's place once the inputs are left out", q_rank, 1, tol=2)
inc, exc = analogy(" king", " man", " woman", table=WTE)
print("GPT-2's table   including the inputs:", inc)
print("                leaving them out:    ", exc)
check("GPT-2's table: the nearest, inputs included", inc[:2], [tuple(x) for x in [[' king', 0.7758], [' queen', 0.7085]]])
check("GPT-2's table: the nearest, inputs left out", exc[0], tuple([' queen', 0.7085]))

### 8. Where is each word? (Plate XII)
Attention (Lecture 20) mixes words without knowing their order, so GPT-2 adds a second arrow to each word: a
**position arrow**, one per place in the text, 1,024 of them (`wpe`). The first is long; the rest are short. Drawn on
their three main directions (the three along which the 1,024 arrows spread most, found by a singular value
decomposition), they make one smooth curve: nearby places get nearby arrows.

In [ ]:
print("lengths of the first 13 position arrows:", np.round(WPE[:13].norm(dim=-1).tolist(), 3))
check("position arrows' lengths (largest difference)", (WPE[:13].norm(dim=-1).cpu() - torch.tensor([9.876, 5.192, 4.585, 4.336, 4.181, 4.061, 3.976, 3.908, 3.851, 3.808, 3.773, 3.737, 3.707])).abs().max().item(), 0, tol=1e-3)
P = WPE - WPE.mean(0)
U, Sg, Vt = torch.linalg.svd(P, full_matrices=False)
share = (Sg[:3] ** 2 / (Sg ** 2).sum()).tolist()
print("share of the spread on the three main directions:", np.round(share, 4))
check("the three directions' shares (largest difference)", max(abs(a - b) for a, b in zip(share, [0.3341, 0.3235, 0.2462])), 0, tol=1e-3)
xyz = (P @ Vt[:3].T).cpu().numpy()
fig = plt.figure(figsize=(5, 4.2)); ax = fig.add_subplot(projection="3d")
ax.set_facecolor(PAPER); ax.plot(*xyz.T, color=INK, lw=1)
for p in (0, 128, 256, 512, 768, 1023):
    ax.scatter(*xyz[p], color=INK, s=12); ax.text(*xyz[p], f" {p}", fontsize=8)
ax.set_title("1,024 position arrows, on their three main directions", fontsize=10)
for a in (ax.xaxis, ax.yaxis, ax.zaxis):
    a.set_pane_color(PAPER); a.set_ticklabels([])
plt.show()

## Part 3 · The running total

### 9. The running total begins (Plate XIII)
Each word's first **running total** is its word arrow plus its position arrow: 768 numbers. Every block that follows
reads this total and *adds* something to it; nothing is ever overwritten. (The papers call it the *residual stream*.)

In [ ]:
x0 = WTE[ids] + WPE[:len(ids)]
print(f"the last word's first total: length {x0[-1].norm():.3f}")
check("the last word's first running total, its length", x0[-1].norm().item(), 4.634, tol=1e-3)

### 10. LayerNorm (Plate XVIII)
Before reading the total, a block **LayerNorm**s it: subtract the 768 numbers' average, divide by their spread (the
standard deviation), then multiply each number by a learned *gain* and add a learned *shift*. The result no longer
depends on how big the total is: scale it ten times and the output does not move.

In [ ]:
def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    return (x - mu) / torch.sqrt(((x - mu) ** 2).mean(-1, keepdim=True) + eps) * g + b

g1, b1 = SD["h.0.ln_1.weight"], SD["h.0.ln_1.bias"]
a, a10 = layer_norm(x0[-1], g1, b1), layer_norm(10 * x0[-1], g1, b1)
print("block 1's LayerNorm of the last word, first 6 numbers:", np.round(a[:6].tolist(), 4))
print("...and of ten times the total:                      ", np.round(a10[:6].tolist(), 4))
check("scaling the total by 10 moves the output by (largest change)", (a - a10).abs().max().item(), 0, tol=1e-3)

### 11. A smooth gate (Plate XVII)
Lecture 1's gate, **ReLU**, lets positive numbers through and turns negatives to 0. GPT-2 uses a smooth version,
**GELU**: almost 0 for very negative inputs, almost the input for positive ones, and a small dip in between.

In [ ]:
def gelu(x):                                   # GPT-2's form of it
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

xs = torch.linspace(-4, 3, 7001)
ys = gelu(xs)
i = ys.argmin()
print(f"GELU's lowest point: {ys[i]:.3f} at {xs[i]:.3f}")
check("GELU's lowest value", round(ys[i].item(), 2), -0.17)
check("where it is lowest", round(xs[i].item(), 2), -0.75)
fig, ax = plt.subplots(figsize=(5, 2.8))
ax.plot(xs, torch.relu(xs), color=INK2, lw=1.2, ls="--", label="ReLU")
ax.plot(xs, ys, color=INK, lw=1.8, label="GELU")
ax.axhline(0, color=INK2, lw=0.5); ax.legend(); ax.set_xlabel("input"); ax.set_ylabel("output"); plt.show()

### 12. Inside a block (Plate XVI)
Each of GPT-2's twelve blocks has two side roads off the main road (the running total), and each adds its result back:

* **attention**: each word looks back at the words before it and copies some of their information into its own total.
  It is twelve *heads* working side by side; Lecture 20 takes one apart. Here it is in ten lines.
* **the neurons** (the *MLP*): 3,072 neurons of Lecture 1, each reading the LayerNormed total, gated by GELU, each
  writing its own arrow back, scaled by how strongly it fires.

`run()` below is the whole model. It keeps every intermediate number, so the rest of the notebook can read them.

In [ ]:
@torch.no_grad()
def run(ids, allow=None, zero_heads=()):
    """One pass of GPT-2 over a list of token ids; every intermediate number is kept in a dict."""
    ids = torch.as_tensor(ids, device=DEV)
    T = len(ids)
    c = {"tok": WTE[ids], "pos": WPE[:T]}
    x = c["tok"] + c["pos"]                                                     # the running total begins
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=DEV))           # no peeking at later words
    if allow is not None:
        mask = mask & allow
    keep = {k: [] for k in ("resid", "q", "k", "v", "pattern", "head_out", "attn_out", "mlp_post", "mlp_out")}
    for l in range(12):
        p = lambda n: SD[f"h.{l}.{n}"]
        keep["resid"].append(x)
        # attention: 12 heads, each with a 64-number query, key and value per word
        h = layer_norm(x, p("ln_1.weight"), p("ln_1.bias"))
        q, k, v = (h @ p("attn.c_attn.weight") + p("attn.c_attn.bias")).split(768, -1)
        q, k, v = (t.view(T, 12, 64).transpose(0, 1) for t in (q, k, v))
        s = (q @ k.transpose(-1, -2) / 8).masked_fill(~mask, float("-inf"))     # how well each query matches each key
        a = torch.softmax(s, -1)                                                # ...as shares that add up to 1
        ho = torch.einsum("htd,hdm->htm", a @ v, p("attn.c_proj.weight").view(12, 64, 768))   # each head's addition
        for zl, zh in zero_heads:
            if zl == l:
                ho[zh] = 0
        att = ho.sum(0) + p("attn.c_proj.bias")
        x = x + att
        # the 3,072 neurons
        post = gelu(layer_norm(x, p("ln_2.weight"), p("ln_2.bias")) @ p("mlp.c_fc.weight") + p("mlp.c_fc.bias"))
        mo = post @ p("mlp.c_proj.weight") + p("mlp.c_proj.bias")
        x = x + mo
        for key, val in zip(("q", "k", "v", "pattern", "head_out", "attn_out", "mlp_post", "mlp_out"), (q, k, v, a, ho, att, post, mo)):
            keep[key].append(val)
    keep["resid"].append(x)
    c.update({k: torch.stack(v) for k, v in keep.items()})
    c["final"] = layer_norm(x, SD["ln_f.weight"], SD["ln_f.bias"])
    c["logits"] = c["final"] @ WTE.T                   # the scores (Part 4)
    return c

c = run(ids)
print("kept for every block:", {k: tuple(v.shape) for k, v in c.items() if k in ("resid", "pattern", "mlp_post")})
fire = [(c["mlp_post"][l, -1] > 0).sum().item() for l in range(12)]
print("neurons firing (above 0) at the last word, block by block:", fire)
check("neurons firing in each block", fire, [290, 950, 568, 327, 444, 507, 529, 505, 541, 525, 558, 663])

### 13. Twelve blocks, one running total (Plate XIV)
The last word's total grows as the blocks add to it: from 4.6 to 465.5.

In [ ]:
norms = c["resid"][:, -1].norm(dim=-1).tolist()
print("the last word's total, its length after each block:", np.round(norms, 1))
check("the lengths, block by block (largest difference)", max(abs(a - b) for a, b in zip(norms, [4.634, 53.271, 55.928, 58.544, 62.761, 68.076, 73.332, 81.483, 99.404, 124.191, 160.342, 238.28, 465.535])), 0, tol=0.01)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(range(13), norms, color=INK, marker="o", ms=4); ax.set_yscale("log")
ax.set_xticks(range(13), ["start"] + [str(l) for l in range(1, 13)]); ax.set_xlabel("after block")
ax.set_ylabel("length (log scale)"); ax.set_title("The running total grows", fontsize=10); plt.show()

### 14. It adds up, exactly (Plate XV)
Since every block only *adds*, the final total is exactly the sum of 26 pieces: the word arrow, the position arrow,
and each block's attention and neurons.

In [ ]:
pieces = [("word arrow", c["tok"][-1]), ("position arrow", c["pos"][-1])]
for l in range(12):
    pieces += [(f"attention {l + 1}", c["attn_out"][l, -1]), (f"neurons {l + 1}", c["mlp_out"][l, -1])]
total = pieces[0][1].clone()
for _, v in pieces[1:]:
    total = total + v
gap = (total - c["resid"][12, -1]).abs().max().item()
print(f"{len(pieces)} pieces; their sum against the final total: largest difference {gap}")
check("26 pieces add up to the final total", gap, 0.0)
print("the longest pieces:", sorted([(n, round(v.norm().item(), 1)) for n, v in pieces], key=lambda t: -t[1])[:4])

## Part 4 · From the total to chances

### 15. The last LayerNorm (Plate XVIII)
The final total goes through one more LayerNorm before it is read. Its numbers are wildly uneven: a few are in the
hundreds.

In [ ]:
xf = c["resid"][12, -1]
mu, sd_ = xf.mean().item(), xf.std(unbiased=False).item()
print(f"the final total: average {mu:.4f}, spread {sd_:.4f}; largest {xf.max():.2f} (number {xf.argmax()}), smallest {xf.min():.2f} (number {xf.argmin()})")
check("its average", mu, -0.0403, tol=1e-3)
check("its spread", sd_, 16.7985, tol=2e-3)
check("its largest number is number", xf.argmax().item(), 496)
check("its smallest number is number", xf.argmin().item(), 373)
check("after LayerNorm, the first 8 (largest difference)", (c["final"][-1, :8].cpu() - torch.tensor([-0.2547, -0.5507, 0.8027, 0.4446, 0.1121, -0.3307, -0.5446, -0.9905])).abs().max().item(), 0, tol=2e-4)

### 16. One dot product (Plate XIX)
To score *walk*, multiply the final (LayerNormed) total by walk's own row of the word table, number by number, and add
the 768 products. The same table that turned tokens into arrows now turns the total back into words.

In [ ]:
fin = c["final"][-1]
prods = fin * WTE[2513]
score = prods.sum().item()
print(f"walk's score: {score:.4f}; {(prods > 0).sum().item()} products are positive, adding {prods[prods > 0].sum():.2f}; "
      f"the rest add {prods[prods < 0].sum():.2f}")
check("walk's score", score, -109.6762, tol=2e-3)
check("positive products", (prods > 0).sum().item(), 453, tol=2)
big = prods.abs().argsort(descending=True)[:4].tolist()
print("the four biggest products are at numbers", big, "->", np.round(prods[big].tolist(), 2))
check("the four huge numbers", big, [496, 36, 430, 314])

In [ ]:
# Those four numbers give every word nearly the same score, because every row of the word table holds nearly the same
# value there. The real choice is made by the other 764.
L = c["logits"][-1]
from_big = WTE[:, big] @ fin[big]
print(f"from the 4 huge numbers alone, every word scores {from_big.mean():.2f} on average (spread {from_big.std():.2f})")
print(f"walk: {from_big[2513]:.2f} from the four, {L[2513] - from_big[2513]:.2f} from the other 764")
check("the four huge numbers' average score", from_big.mean().item(), -117.2694, tol=0.01)
check("walk's score from the other 764", (L[2513] - from_big[2513]).item(), 6.1354, tol=0.01)

### 17. 50,257 scores (Plate XX)
Do the same for every row of the word table: one score per token. That is one big multiplication, `final @ WTE.T`,
which `run()` already did.

In [ ]:
print(f"{len(L):,} scores: highest {L.max():.4f} ({decode([L.argmax()])!r}), lowest {L.min():.4f}, average {L.mean():.4f}, spread {L.std():.4f}")
check("the highest score is walk's", decode([L.argmax()]), TARGET)
check("the lowest score", L.min().item(), -146.7046, tol=2e-3)
check("the average score", L.mean().item(), -128.9732, tol=2e-3)
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.hist(L.cpu().numpy(), bins=np.arange(-147, -109, 0.25), color=INK, histtype="stepfilled")
ax.annotate("walk", (L[2513].item(), 0), xytext=(L[2513].item() - 1.5, 900), arrowprops=dict(arrowstyle="->", color=INK))
ax.set_xlabel("score"); ax.set_ylabel("tokens"); ax.set_title("Every token's score after “…went for a”", fontsize=10); plt.show()

### 18. From scores to chances (Plate XXI) — and Plate I's answer
**Softmax** turns scores into chances: raise *e* to each score, then divide by the sum, so the chances add up to 1.
Only differences between scores matter, so we first subtract the highest (the numbers stay small enough for a
computer).

In [ ]:
def softmax(scores, temperature=1.0):
    e = torch.exp((scores - scores.max()) / temperature)
    return e / e.sum()

P = softmax(L)
top = P.topk(10)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {decode([i])!r:10} {100 * p:5.2f}%  " + "█" * round(60 * p))
print(f"the top 10 hold {100 * top.values.sum():.1f}% of the chance")
check("the chance of walk", P[2513].item(), 0.579746, tol=1e-4)
check("the chance of run", P[1057].item(), 0.05377, tol=1e-4)
check("the ten likeliest words", [decode([i]) for i in top.indices.tolist()], [' walk', ' run', ' jog', ' stroll', ' drive', ' swim', ' quick', ' hike', ' ride', ' few'])
check("Plate I: walk, as a percentage", round(100 * P[2513].item(), 1), 58.0)

### 19. A guess at every word (Plate III)
GPT-2 makes a guess after *every* token, not only the last: each word's final total is scored against the whole table.

In [ ]:
Pall = torch.softmax(c["logits"], -1)
nxt = ids[1:] + [2513]
p_next = [Pall[i, n].item() for i, n in enumerate(nxt)]
rank_next = [int((Pall[i] > Pall[i, n]).sum()) + 1 for i, n in enumerate(nxt)]
for i in range(len(ids)):
    t3 = Pall[i].topk(3)
    guess = ", ".join(f"{decode([j])!r} {100 * p:.0f}%" for p, j in zip(t3.values.tolist(), t3.indices.tolist()))
    print(f"after {decode(ids[:i + 1])[-22:]!r:26} guesses {guess:42} next was {decode([nxt[i]])!r} ({100 * p_next[i]:.2f}%, rank {rank_next[i]})")
check("the chance given to each next word (largest difference)", max(abs(a - b) for a, b in zip(p_next, [0.0253, 0.1559, 0.00105, 0.14392, 0.11496, 0.50599, 0.17925, 0.02563, 0.12451, 0.0572, 0.02315, 0.72228, 0.57975])), 0, tol=2e-4)
check("each next word's rank", rank_next, [5, 1, 144, 1, 1, 1, 1, 5, 2, 2, 10, 1, 1])

### 20. Surprise, counted in nats (Plate IV)
How surprised was GPT-2 by each word? **Surprise** = −ln(the chance it gave the word that really came next): 0 for a
word it was sure of, large for a word it thought unlikely. Its average over a text is the **loss**, the number
training pushes down.

In [ ]:
surprise = [-math.log(p) for p in p_next[:12]]                 # the 12 words after "It"
print("surprise of each word (nats):", np.round(surprise, 2))
print(f"average: {np.mean(surprise):.3f} nats")
check("the average surprise on this sentence", np.mean(surprise), 2.633, tol=2e-3)
fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.bar(range(12), surprise, color=INK); ax.set_xticks(range(12), [decode([n]) for n in nxt[:12]], fontsize=8, rotation=30)
ax.set_ylabel("nats"); ax.set_title("How surprising each word was", fontsize=10); plt.show()

### 21. Again and again (Plate XXII)
To write, take a word, add it to the text, and run the model again. Taking the likeliest word every time is
**greedy**; drawing a word at random by its chance is **sampling**.

In [ ]:
def generate(ids, n=25, greedy=True, seed=0):
    gen = torch.Generator().manual_seed(seed)
    cur = list(ids)
    for _ in range(n):
        p = torch.softmax(run(cur)["logits"][-1], -1)
        cur.append(int(p.argmax()) if greedy else int(torch.multinomial(p.cpu(), 1, generator=gen)))
    return decode(cur[len(ids):])

g = generate(ids)
print("greedy:", repr(SENTENCE + g))
check("the greedy continuation", g, ' walk. We were walking along the road, and I saw a man walking down the road. He was wearing a white shirt')
for seed in (1, 2, 3):
    print(f"sample {seed}:", repr(SENTENCE + generate(ids, greedy=False, seed=seed)))

### 22. What one word costs (Plate XXIII)
Each new word needs every weight of every block once: a **multiply-add** is one multiplication and one addition.
Count them from the shapes.

In [ ]:
per_block = sum(SD[f"h.0.{n}"].numel() for n in ("attn.c_attn.weight", "attn.c_proj.weight", "mlp.c_fc.weight", "mlp.c_proj.weight"))
last_step = WTE.numel()                                      # 50,257 dot products of 768
total = 12 * per_block + last_step
print(f"one block: {per_block:,}; twelve: {12 * per_block:,}; the last step: {last_step:,}; in all: {total:,} per word")
print(f"the last step, scoring every word, is {100 * last_step / total:.0f}% of it")
check("multiply-adds in one block", per_block, 7077888)
check("multiply-adds per word", total, 123532032)
check("the last step's share", round(100 * last_step / total), 31)
if DEV == "cuda":
    run(ids); torch.cuda.synchronize(); t = time.time()
    for _ in range(20):
        run(ids)
    torch.cuda.synchronize()
    print(f"on this GPU, our hand-written model reads the 13-word sentence in {(time.time() - t) / 20 * 1000:.1f} ms "
          "(the lecture's T4: 9.8 ms)")

## Part 5 · Reading the answer as it forms

### 23. Watching the answer form (Plate XXIV)
The **logit lens** (nostalgebraist, 2020): read the running total *after any block* as if it were the final one,
through the last LayerNorm and the word table. What would GPT-2 say if it stopped there?

In [ ]:
def lens(r):
    return layer_norm(r, SD["ln_f.weight"], SD["ln_f.bias"]) @ WTE.T

race = {w: [] for w in (" walk", " stroll", " ride", " lot", " couple")}
lens_p, lens_rank, lens_top = [], [], []
for l in range(13):
    p = torch.softmax(lens(c["resid"][l, -1]), -1)
    lens_p.append(p[2513].item()); lens_rank.append(int((p > p[2513]).sum()) + 1); lens_top.append(decode([p.argmax()]))
    for w in race:
        race[w].append(p[encode(w)[0]].item())
    t2 = p.topk(2)
    print(f"{'start' if l == 0 else 'after block ' + str(l):15} " + ", ".join(f"{decode([j])!r} {100 * v:.0f}%" for v, j in zip(t2.values.tolist(), t2.indices.tolist()))
          + f"     walk {100 * p[2513]:.2f}% (rank {lens_rank[-1]})")
check("walk's chance through the lens (largest difference)", max(abs(a - b) for a, b in zip(lens_p, [0.0, 4.2e-05, 6e-05, 5.3e-05, 8.3e-05, 0.000173, 0.001141, 0.003304, 0.011946, 0.048631, 0.106755, 0.434677, 0.579748])), 0, tol=1e-4)
check("walk's rank through the lens", lens_rank, [29426, 630, 354, 289, 151, 105, 52, 32, 12, 4, 3, 2, 1])
check("the top word after each block", lens_top, [' mathemat', ' very', ' lot', ' lot', ' lot', ' lot', ' lot', ' couple', ' couple', ' ride', ' stroll', ' stroll', ' walk'])
check("stroll after block 10", race[" stroll"][10], 0.615978, tol=1e-4)
fig, ax = plt.subplots(figsize=(6.5, 2.8))
for w, ls in zip(race, ["-", "--", ":", "-.", (0, (1, 3))]):
    ax.plot(range(13), race[w], color=INK, ls=ls, lw=1.6, label=w.strip())
ax.set_xticks(range(13), ["start"] + [str(l) for l in range(1, 13)]); ax.set_xlabel("after block"); ax.set_ylabel("chance")
ax.legend(ncol=5, fontsize=8, loc="upper left"); ax.set_title("The race, read through the lens", fontsize=10); plt.show()

### 24. Walk against stroll (Plate XXV)
After block 11, stroll led. Who pushed walk ahead? Since the final total is a sum of 26 pieces, and the last
LayerNorm is a fixed straight-line map once its spread is frozen at the final total's, each piece's **push** on
(walk's score − stroll's score) can be read on its own, and the pushes add up exactly.

In [ ]:
sigma = torch.sqrt(((xf - xf.mean()) ** 2).mean() + 1e-5)
gain, shift = SD["ln_f.weight"], SD["ln_f.bias"]

def push(v, u):
    """What a piece v of the final total adds to the score difference along u (u = one word's row minus another's)."""
    return (((v - v.mean()) / sigma * gain) @ u).item()

u_ws = WTE[2513] - WTE[encode(" stroll")[0]]
ws = [push(v, u_ws) for _, v in pieces]
ws_bias = (shift @ u_ws).item()
print(f"walk − stroll: the 26 pushes add to {sum(ws):.4f}, the last LayerNorm's shift adds {ws_bias:.4f}: {sum(ws) + ws_bias:.4f}")
print(f"the scores' real difference: {(L[2513] - L[encode(' stroll')[0]]).item():.4f}")
for (n, _), v in sorted(zip(pieces, ws), key=lambda t: -abs(t[1]))[:5]:
    print(f"   {n:15} {v:+.3f}")
check("walk − stroll, every push added up", sum(ws) + ws_bias, 2.6546, tol=2e-3)
check("the scores' own difference, walk − stroll", (L[2513] - L[encode(" stroll")[0]]).item(), 2.6546, tol=2e-3)
check("each piece's push (largest difference)", max(abs(a - b) for a, b in zip(ws, [-0.037, 0.0386, 0.1793, 0.1192, -0.0476, 0.0814, -0.1276, -0.0225, -0.0803, 0.127, -0.075, -0.1585, 0.0268, -0.0388, -0.1153, -0.325, -0.1005, -0.1578, -0.2257, -0.2186, -0.2945, -0.0772, -0.4045, 0.5359, -0.1341, 1.4977])), 0, tol=2e-3)

### 25. Who built walk? (Plate XXVI)
The same pushes, now on walk's score above the *average* word's: the 26 pieces as a staircase.

In [ ]:
u_wm = WTE[2513] - WTE.mean(0)
wm = [push(v, u_wm) for _, v in pieces]
wm_bias = (shift @ u_wm).item()
print(f"walk above the average word: 26 pushes {sum(wm):.2f} + the shift {wm_bias:.2f} = {sum(wm) + wm_bias:.2f}")
check("walk above the average word", sum(wm) + wm_bias, 19.2971, tol=2e-3)
check("the last LayerNorm's shift", wm_bias, 2.0667, tol=1e-3)
check("each piece's push (largest difference)", max(abs(a - b) for a, b in zip(wm, [-0.0377, 0.0078, 0.5113, 0.3767, 0.095, 0.1625, 0.0466, 0.0932, 0.12, 0.2462, 0.1511, 0.1301, 0.2747, 0.2011, 0.2623, 0.3975, 0.4512, 0.5978, 0.6574, 1.1858, 0.3862, 2.1397, 1.195, 3.1132, 1.8332, 2.6327])), 0, tol=2e-3)
steps = np.cumsum([0] + wm)
fig, ax = plt.subplots(figsize=(8, 3))
for i, v in enumerate(wm):
    ax.bar(i, v, bottom=steps[i], color=INK if v > 0 else PAPER, edgecolor=INK, width=0.8)
ax.bar(26, wm_bias, bottom=steps[-1], color=INK2, width=0.8)
ax.set_xticks(range(27), [n.replace("attention", "att").replace("neurons", "neu").replace(" arrow", "") for n, _ in pieces] + ["shift"],
              rotation=90, fontsize=7)
ax.set_ylabel("walk − average"); ax.set_title("Who built walk: 26 pushes, then the last LayerNorm's shift", fontsize=10); plt.show()

### 26. Neurons, read through the words (Plate XXVII)
Go one level down: each neuron writes its own arrow, scaled by how strongly it fires. Its push on walk is that
arrow's push, times its firing. And its arrow alone, read through the word table, says which words it promotes.
(Blocks and neurons are counted from 1 here, as on the plates; in code, from 0.)

In [ ]:
rows = []
for l in range(12):
    W_out = SD[f"h.{l}.mlp.c_proj.weight"]                            # 3,072 x 768: neuron n writes row n
    per = ((W_out - W_out.mean(-1, keepdim=True)) * gain) @ u_wm / sigma
    rows.append(c["mlp_post"][l, -1] * per)
D = torch.stack(rows)                                                 # 12 x 3,072 pushes on walk
order = D.flatten().argsort(descending=True)
best = [divmod(int(i), 3072) for i in order[:4]]
print("the four neurons pushing walk hardest:", [(f"block {l + 1}, no. {n}", round(D[l, n].item(), 3)) for l, n in best])
print(f"all 36,864 neurons together push {D.sum():.2f}; {(D > 0.05).sum().item()} push by more than 0.05")
check("the four hardest pushers (block, neuron, from 0)", [list(b) for b in best], [[11, 611], [8, 1333], [11, 386], [11, 896]])
check("their pushes (largest difference)", max(abs(D[l, n].item() - e) for (l, n), e in zip(best, [0.5278, 0.444, 0.4212, 0.3799])), 0, tol=2e-3)
check("neurons pushing by more than 0.05", (D > 0.05).sum().item(), 130, tol=1)

def reads_as(l, n, k=8):
    """A neuron's arrow, read through the word table: the words it promotes most."""
    w = SD[f"h.{l}.mlp.c_proj.weight"][n]
    s = ((w - w.mean()) * gain) @ WTE.T
    return [decode([i]) for i in (s - s.mean()).topk(k).indices.tolist()]

for l, n in [(8, 1333), (11, 386), (11, 611)]:
    print(f"block {l + 1}, no. {n}: fires {c['mlp_post'][l, -1, n]:.2f} here; promotes {reads_as(l, n)}")
check("block 9, no. 1333 promotes", reads_as(8, 1333, 4), ['uras', ' lot', ' few', ' couple'])
check("block 12, no. 386 promotes", reads_as(11, 386, 4), [' contest', ' bout', ' game', ' expedition'])

Words a neuron promotes are one reading; the texts that make it fire are another. We read 38,400 words of Wikipedia
(**WikiText-103**'s validation set: 300 passages of 128 tokens, evenly spaced) and keep each neuron's strongest
moments. The text is cut with *our* tokenizer, so the passages are the lecture's, token for token.

In [ ]:
import pyarrow.parquet as pq
fetch("https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/wikitext-103-raw-v1/validation-00000-of-00001.parquet",
      "wikitext_val.parquet", "WikiText-103's validation set")
t = time.time()
WIKI = encode("".join(pq.read_table("wikitext_val.parquet").column("text").to_pylist()))
print(f"{len(WIKI):,} tokens, cut by our own tokenizer in {time.time() - t:.0f} s")

watch = {"L8N1333": (8, 1333), "L11N386": (11, 386), "L9N400": (9, 400)}
step = len(WIKI) // 300
acts = {k: [] for k in watch}
for k in range(300):
    m = run(WIKI[k * step: k * step + 128])["mlp_post"]
    for key, (l, n) in watch.items():
        acts[key].append(m[l, :, n].cpu())
for key, (l, n) in watch.items():
    A = torch.cat(acts[key])
    j = int(A.argmax()); pos = (j // 128) * step + j % 128
    print(f"block {l + 1}, no. {n}: fires above 0 on {100 * (A > 0).float().mean():.1f}% of words; strongest {A.max():.3f} at "
          f"…{decode(WIKI[pos - 12:pos + 1])!r} → {decode(WIKI[pos + 1:pos + 3])!r}")
    check(f"block {l + 1}, no. {n}: its strongest firing", A.max().item(), {'L8N1333': {'max': 5.1208, 'frac': 0.0352, 'next': ' lot of'}, 'L11N386': {'max': 4.3528, 'frac': 0.1877, 'next': ' exercise ('}, 'L9N400': {'max': 4.385, 'frac': 0.0964, 'next': ' to Manila'}}[key]["max"], tol=5e-3)
    check(f"block {l + 1}, no. {n}: what came next", decode(WIKI[pos + 1:pos + 3]), {'L8N1333': {'max': 5.1208, 'frac': 0.0352, 'next': ' lot of'}, 'L11N386': {'max': 4.3528, 'frac': 0.1877, 'next': ' exercise ('}, 'L9N400': {'max': 4.385, 'frac': 0.0964, 'next': ' to Manila'}}[key]["next"])

### 27. The lens across 25,600 words (Plates XXVIII–XXIX)
One sentence can mislead. Read 200 passages of 128 tokens through the lens after every block, and count how often
the lens's top word is already the model's final top word, how far its chances are from the final ones (the **KL
divergence**, in nats), and what chance it gives the word that really came next.

In [ ]:
step = len(WIKI) // 200
agree, kl, ce, pt = (torch.zeros(13) for _ in range(4))
for k in range(200):
    w = torch.tensor(WIKI[k * step: k * step + 129], device=DEV)
    cc = run(w[:-1]); y = w[1:]
    Pf = torch.log_softmax(cc["logits"], -1)
    for l in range(13):
        Q = torch.log_softmax(lens(cc["resid"][l]), -1)
        agree[l] += (Q.argmax(-1) == Pf.argmax(-1)).float().sum().cpu()
        kl[l] += (Pf.exp() * (Pf - Q)).sum().cpu()
        ce[l] += -Q.gather(-1, y[:, None]).sum().cpu()
        pt[l] += Q.gather(-1, y[:, None]).exp().sum().cpu()
n = 200 * 128
agree, kl, ce, pt = (v / n for v in (agree, kl, ce, pt))
print("after block:          ", "  ".join(f"{l:>5}" for l in range(13)))
print("top word agrees:      ", "  ".join(f"{v:5.2f}" for v in agree.tolist()))
print("KL from the final:    ", "  ".join(f"{v:5.1f}" for v in kl.tolist()))
print("chance of the truth:  ", "  ".join(f"{v:5.3f}" for v in pt.tolist()))
for name, ours, theirs in (("agreement", agree, [0.0097, 0.0745, 0.0785, 0.0838, 0.1021, 0.1182, 0.1429, 0.1729, 0.2336, 0.3067, 0.4031, 0.5473, 1.0]), ("KL", kl, [49.7477, 7.1053, 6.1274, 5.7107, 5.5046, 5.2471, 4.877, 4.5075, 3.9153, 3.4737, 2.8821, 1.4958, 0.0]), ("loss", ce, [54.0243, 11.4133, 10.5042, 10.1121, 9.9142, 9.6852, 9.2989, 8.9369, 8.2692, 7.5988, 6.8189, 5.4159, 4.0641]), ("chance of the truth", pt, [0.0074, 0.0209, 0.026, 0.0309, 0.0384, 0.0459, 0.0576, 0.0712, 0.1023, 0.1477, 0.1971, 0.2291, 0.1837])):
    check(f"the lens over 25,600 words: {name} (largest difference)", max(abs(a - b) for a, b in zip(ours.tolist(), theirs)), 0, tol=3e-3)
print(f"the last block hedges: the true word's chance is {100 * pt[11]:.1f}% after block 11, {100 * pt[12]:.1f}% at the end")
check("the truth's chance falls in the last block", bool(pt[12] < pt[11]), True)
fig, ax = plt.subplots(1, 2, figsize=(9, 2.8))
ax[0].plot(range(13), agree, color=INK, marker="o", ms=3); ax[0].set_title("lens top word = final top word", fontsize=10)
ax[1].plot(range(13), pt, color=INK, marker="o", ms=3); ax[1].set_title("chance given to the true next word", fontsize=10)
for a in ax:
    a.set_xticks(range(13), ["s"] + [str(l) for l in range(1, 13)]); a.set_xlabel("after block")
plt.tight_layout(); plt.show()

### 28. The lens everywhere (Plate XXX)
The lens's top guess at every word of our sentence, after every block.

In [ ]:
grid = [[decode([int(i)]) for i in torch.softmax(lens(c["resid"][l]), -1).argmax(-1).tolist()] for l in range(13)]
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.set_xlim(-0.5, 12.5); ax.set_ylim(12.5, -0.5)
for l in range(13):
    for i in range(13):
        ax.text(i, l, grid[l][i].replace("\n", "⏎")[:9], ha="center", va="center", fontsize=7,
                fontweight="bold" if grid[l][i] == grid[12][i] else "normal")
ax.set_xticks(range(13), [decode([i]) for i in ids], fontsize=8); ax.xaxis.tick_top()
ax.set_yticks(range(13), ["start"] + [f"block {l}" for l in range(1, 13)], fontsize=8)
ax.set_title("The lens's top guess (bold: already the final guess)", fontsize=10, pad=24); plt.show()
same = sum(grid[l][i] == g for l in range(13) for i, g in enumerate([['theless', ' livest', ' livest', ' warm', ' mathemat', ' sunny', ' day', ' mathemat', ' mathemat', ' mathemat', ' mathemat', ' destro', ' mathemat'], ['\n', ' also', ' very', ' warm', ' which', ' sunny', 'dream', ' which', ' far', "'ve", ' ahead', ' the', ' very'], [',', ' also', ' very', ' warm', ' although', ' sunny', 'dream', ' although', 'bered', "'ve", ' ahead', ' example', ' lot'], [',', ' also', ' very', ' warm', ' albeit', ' sunny', 'dream', ' although', 'bered', "'ve", ' ahead', ' the', ' lot'], [',', ' also', ' very', ' warm', ' albeit', ' sunny', 'dream', ' which', 'bered', "'ve", ' ahead', ' the', ' lot'], [',', ' clear', ' very', ' warm', ' albeit', ' sunny', 'dream', ' which', 'bered', "'ll", ' ahead', ' the', ' lot'], [',', ' clear', ' very', ' warm', ' albeit', ' sunny', 'dream', ' but', 'bered', "'ll", ' ahead', ' the', ' lot'], [',', ' also', ' great', ' warm', ' calm', ' weather', 'dream', ' which', ' far', "'ll", ' ahead', ' lunch', ' couple'], [',', ' clear', ' great', 'est', ' warm', ' sunny', 'dream', ' but', ' far', "'ll", ' downstairs', ' lunch', ' couple'], [',', ' a', ' great', ' warm', ' sunny', ' summer', ',', ' but', ' I', "'ll", ' downstairs', ' dinner', ' ride'], [',', ' a', ' great', ' breeze', ' sunny', ' afternoon', ',', ' but', ' I', ' walked', ' upstairs', ' lunch', ' stroll'], [',', ' a', ' very', ' summer', ' sunny', ' day', ' in', ' and', ' I', ' walked', ' upstairs', ' lunch', ' stroll'], [' is', ' a', ' very', ',', ' sunny', ' day', ',', ' and', ' I', ' were', ' to', ' a', ' walk']][l]))
check("cells that match the lecture's grid", same, 169)

## Your turn

Each exercise has a one-line hint. Worked solutions are at the end.

**1. The weather.** Change *warm, sunny* to *cold, rainy*. What chance does walk get now?
*Hint: `run(encode(...))["logits"][-1]`, then `softmax`.*

In [ ]:
# your code here

**2. Temperature.** Divide the scores by a *temperature* before softmax. What chance does walk get at temperature 0.5,
and at 2? How many words then share the top 90% of the chance?
*Hint: `softmax(L, temperature=0.5)`; sort the chances and count until they pass 0.9.*

In [ ]:
# your code here

**3. Without the four huge numbers.** Score every word from the other 764 numbers only (leave out 496, 36, 430 and 314).
How closely do these scores follow the full ones?
*Hint: zero those four numbers of `fin`, multiply by `WTE.T`, and compare with `np.corrcoef`.*

In [ ]:
# your code here

**4. When does *day* win?** At the word *sunny*, after which block does the lens first pick *day*?
*Hint: the lens grid of Step 28, column 5 (counting from 0).*

In [ ]:
# your code here

**5. Walk against run.** Run was the final runner-up. Add up the 26 pushes on (walk − run). Which piece pushes walk
over run the most?
*Hint: `push(v, WTE[2513] - WTE[encode(" run")[0]])` for each piece, plus `shift @ u`.*

In [ ]:
# your code here

**6. Your own analogy.** Try *paris − france + italy* in GPT-2's word table and in your own word2vec vectors.
*Hint: GPT-2's tokens need their leading space: `" Paris"`; your vectors use lower-case words.*

In [ ]:
# your code here

## Solutions

**1. The weather.**

In [ ]:
p_cold = torch.softmax(run(encode("It was a cold, rainy day, so we went for a"))["logits"][-1], -1)
print(f"cold, rainy: walk {100 * p_cold[2513]:.1f}% (warm, sunny: {100 * P[2513]:.1f}%)")
check("walk on a cold, rainy day", p_cold[2513].item(), 0.64167, tol=2e-4)
print("The weather barely matters: it is “went for a” that makes the walk (Lecture 20 finds the words that do).")

**2. Temperature.**

In [ ]:
for T_ in (0.5, 1, 2):
    p = softmax(L, temperature=T_)
    k = int((p.sort(descending=True).values.cumsum(0) < 0.9).sum()) + 1
    held = "the likeliest word alone holds" if k == 1 else f"the top {k:,} words hold"
    print(f"temperature {T_}: walk {100 * p[2513]:5.1f}%; {held} 90% of the chance")
check("at temperature 1, walk's chance is the plate's", softmax(L, 1.0)[2513].item(), 0.579746, tol=1e-4)
print("Below 1 the likeliest word takes more; above 1 the chance spreads out.")

**3. Without the four huge numbers.**

In [ ]:
rest = fin.clone(); rest[big] = 0
r = (rest @ WTE.T).cpu().numpy()
corr = np.corrcoef(r, L.cpu().numpy())[0, 1]
print(f"correlation with the full scores: {corr:.3f}; walk is still first: {decode([int(r.argmax())])!r}")
check("correlation of the 764-number scores with the full ones", corr, 0.951, tol=2e-3)

**4. When does *day* win?**

In [ ]:
col = [grid[l][5] for l in range(13)]
first = next(l for l in range(13) if col[l] == " day")
print("at 'sunny', the lens says:", col)
print(f"day first wins after block {first}")
check("the block after which day first wins at 'sunny'", first, 11)

**5. Walk against run.**

In [ ]:
u_wr = WTE[2513] - WTE[encode(" run")[0]]
wr = [push(v, u_wr) for _, v in pieces]
tot = sum(wr) + (shift @ u_wr).item()
print(f"walk − run: {tot:.4f} (the scores' difference: {(L[2513] - L[1057]).item():.4f})")
print("biggest pushes:", sorted([(n, round(v, 3)) for (n, _), v in zip(pieces, wr)], key=lambda t: -t[1])[:3])
check("walk − run, every push added up", tot, 2.3779, tol=2e-3)

**6. Your own analogy.**

In [ ]:
for a, b, c_ in ((" Paris", " France", " Italy"), (" walking", " walk", " swim")):
    inc, exc = analogy(a, b, c_, table=WTE)
    print(f"GPT-2's table, {a!r} − {b!r} + {c_!r}:", [w for w, _ in inc], "   inputs left out:", [w for w, _ in exc])
    inc, exc = analogy(a.strip().lower(), b.strip().lower(), c_.strip().lower())
    print(f"your word2vec, {a.strip().lower()} − {b.strip().lower()} + {c_.strip().lower()}:", [w for w, _ in inc],
          "   inputs left out:", [w for w, _ in exc])

In [ ]:
print(f"The whole notebook ran in {(time.time() - T0) / 60:.1f} minutes.")